# Comparing two classification models using `stambo`


[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Oulu-IMEDS/stambo/main?labpath=notebooks%2FClassification.ipynb)

V1.1.3: © Aleksei Tiulpin, PhD, 2025

This notebook shows an end-to-end example on how one can take a dataset, train two machine learning models, and conduct a statistical test to assess whether the two models are different. We will first use a set of classical metrics (basically the metrics from sklearn). At the end of the tutorial, we will show how one can generate a LaTeX report, and implement a custom metric. 

## Import of necessary libraries

In [1]:
import stambo

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 2025

stambo.__version__

'0.1.6'

## Loading the UCI breast cancer dataset and creating train-test split

In [2]:
X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=SEED, stratify=y)

scaler = StandardScaler()
scaler.fit(Xtr)

Xtr = scaler.transform(Xtr)
Xte = scaler.transform(Xte)

## Training the models

We train a kNN and a logistic regression. Here, we can see that the logistic regression outperformes the kNN. 

In [3]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(Xtr, ytr)
preds_knn = model.predict_proba(Xte)[:, 1]

model = LogisticRegression(C=1e-2, random_state=42)
model.fit(Xtr, ytr)
preds_lr = model.predict_proba(Xte)[:, 1]


auc_knn, auc_lr = roc_auc_score(yte, preds_knn), roc_auc_score(yte, preds_lr)
print(f"kNN AUC: {auc_knn:.4f} / LR AUC: {auc_lr:.4f}")

kNN AUC: 0.9728 / LR AUC: 0.9888


## Statistical testing

As stated in the documentation, the testing routine returns the `dict` of `tuple`. The keys in the dict are the metric tags, and the values are tuples that store the data in the following format:

* p-value $p(H_0 \mid \texttt{data})$
* Empirical value (model 1)
* CI low (model 1)
* CI high (model 1)
* Empirical value (model 2)
* CI low (model 2)
* CI high (model 2)

If you launch the code in Binder, decrease the number of bootstrap iterations (`10000` by default).

In [4]:
testing_result = stambo.compare_models(yte, preds_knn, preds_lr, metrics=("ROCAUC", "AP", "QKappa", "BACC", "MCC"), seed=SEED, n_bootstrap=1000)

Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

If we want to visualize the testing results, they are available in a dict in the format we have described above:

In [5]:
testing_result

{'ROCAUC': {'p_value': 0.10989010989010989,
  'diff': 0.016074628438916494,
  'ci_es': (0.0011614303013786215, 0.03622365794387381),
  'ci_s1': (0.9490493728705037, 0.9925275462036619),
  'ci_s2': (0.9747814271953842, 0.9982587433114521),
  'emp_s1': 0.9727521872035416,
  'emp_s2': 0.9888268156424581},
 'AP': {'p_value': 0.055944055944055944,
  'diff': 0.023144311990304867,
  'ci_es': (0.005028817149312964, 0.04678528920344301),
  'ci_s1': (0.941938772504055, 0.992557217298247),
  'ci_s2': (0.9810928996823938, 0.9989379380439234),
  'emp_s1': 0.9689867972199498,
  'emp_s2': 0.9921311092102547},
 'QKappa': {'p_value': 0.16183816183816183,
  'diff': -0.03208940366959201,
  'ci_es': (-0.0789312628900082, 0.007644376319408607),
  'ci_s1': (0.8538726858185821, 0.9558971346535168),
  'ci_s2': (0.812992125984252, 0.9302711162834225),
  'emp_s1': 0.9081089795260358,
  'emp_s2': 0.8760195758564437},
 'BACC': {'p_value': 0.12387612387612387,
  'diff': -0.02079160957099191,
  'ci_es': (-0.0471810

Most commonly, we though want to visualize them in a report, paper, or a presentation. For that, we can use a function `to_latex`, and get a cut-and-paste `tabular`. To use it in a LaTeX document, one needs to not forget to import booktabs

In [6]:
print(stambo.to_latex(testing_result, m1_name="kNN", m2_name="LR"))

% \usepackage{booktabs} <-- do not forget to have this imported. 
\begin{tabular}{llllll} \\ 
\toprule 
\textbf{Model} & \textbf{ROCAUC} & \textbf{AP} & \textbf{QKappa} & \textbf{BACC} & \textbf{MCC} \\ 
\midrule 
kNN & $0.97$ [$0.95$-$0.99$] & $0.97$ [$0.94$-$0.99$] & $0.91$ [$0.85$-$0.96$] & $0.95$ [$0.91$-$0.97$] & $0.91$ [$0.86$-$0.96$] \\ 
LR & $0.99$ [$0.97$-$1.00$] & $0.99$ [$0.98$-$1.00$] & $0.88$ [$0.81$-$0.93$] & $0.92$ [$0.89$-$0.96$] & $0.88$ [$0.83$-$0.93$] \\ 
\midrule
Effect size & $0.02$ [$0.00$-$0.04]$ & $0.02$ [$0.01$-$0.05]$ & $-0.03$ [$-0.08$-$0.01]$ & $-0.02$ [$-0.05$-$0.00]$ & $-0.03$ [$-0.07$-$0.01]$ \\ 
\midrule
$p$-value & $0.11$ & $0.06$ & $0.16$ & $0.12$ & $0.19$ \\ 
\bottomrule
\end{tabular}


## Own metrics

Sometimes, having default metrics is not enough, and one may want to have some additional metrics. Let us define an F2 score.

In [7]:
from sklearn.metrics import fbeta_score
from functools import partial
from stambo.metrics import Metric

In [8]:
class F2Score(Metric):
    def __init__(self) -> None:
        Metric.__init__(self, partial(fbeta_score, beta=2), int_input=True)

    def __str__(self) -> str:
        return "F2Score"

In [9]:
testing_result = stambo.compare_models(yte, preds_knn, preds_lr, 
                                       ("ROCAUC", "AP", F2Score()),seed=SEED, n_bootstrap=1000)

Bootstrapping:   0%|          | 0/1000 [00:00<?, ?it/s]

In [10]:
print(stambo.to_latex(testing_result, m1_name="kNN", m2_name="LR"))

% \usepackage{booktabs} <-- do not forget to have this imported. 
\begin{tabular}{llll} \\ 
\toprule 
\textbf{Model} & \textbf{ROCAUC} & \textbf{AP} & \textbf{F2Score} \\ 
\midrule 
kNN & $0.97$ [$0.95$-$0.99$] & $0.97$ [$0.94$-$0.99$] & $0.98$ [$0.97$-$0.99$] \\ 
LR & $0.99$ [$0.97$-$1.00$] & $0.99$ [$0.98$-$1.00$] & $0.98$ [$0.97$-$0.99$] \\ 
\midrule
Effect size & $0.02$ [$0.00$-$0.04]$ & $0.02$ [$0.01$-$0.05]$ & $-0.00$ [$-0.01$-$0.01]$ \\ 
\midrule
$p$-value & $0.11$ & $0.06$ & $0.96$ \\ 
\bottomrule
\end{tabular}


In [11]:
testing_result

{'ROCAUC': {'p_value': 0.10989010989010989,
  'diff': 0.016074628438916494,
  'ci_es': (0.0011614303013786215, 0.03622365794387381),
  'ci_s1': (0.9490493728705037, 0.9925275462036619),
  'ci_s2': (0.9747814271953842, 0.9982587433114521),
  'emp_s1': 0.9727521872035416,
  'emp_s2': 0.9888268156424581},
 'AP': {'p_value': 0.055944055944055944,
  'diff': 0.023144311990304867,
  'ci_es': (0.005028817149312964, 0.04678528920344301),
  'ci_s1': (0.941938772504055, 0.992557217298247),
  'ci_s2': (0.9810928996823938, 0.9989379380439234),
  'emp_s1': 0.9689867972199498,
  'emp_s2': 0.9921311092102547},
 'F2Score': {'p_value': 0.9590409590409591,
  'diff': -0.000988531817988858,
  'ci_es': (-0.01017057865486726, 0.010790424857444192),
  'ci_s1': (0.970317290652865, 0.9933774834437086),
  'ci_s2': (0.9725400457665904, 0.9907120743034056),
  'emp_s1': 0.9834254143646409,
  'emp_s2': 0.9824368825466521}}

In [12]:
print(stambo.to_latex(testing_result, m1_name="kNN", m2_name="LR"))

% \usepackage{booktabs} <-- do not forget to have this imported. 
\begin{tabular}{llll} \\ 
\toprule 
\textbf{Model} & \textbf{ROCAUC} & \textbf{AP} & \textbf{F2Score} \\ 
\midrule 
kNN & $0.97$ [$0.95$-$0.99$] & $0.97$ [$0.94$-$0.99$] & $0.98$ [$0.97$-$0.99$] \\ 
LR & $0.99$ [$0.97$-$1.00$] & $0.99$ [$0.98$-$1.00$] & $0.98$ [$0.97$-$0.99$] \\ 
\midrule
Effect size & $0.02$ [$0.00$-$0.04]$ & $0.02$ [$0.01$-$0.05]$ & $-0.00$ [$-0.01$-$0.01]$ \\ 
\midrule
$p$-value & $0.11$ & $0.06$ & $0.96$ \\ 
\bottomrule
\end{tabular}
